# 🎙️ ROTBOTS — The Voice

---

Generates spoken narration for each scene using Edge-TTS (free).

> **🤔** What does a synthetic voice do to trust?

---

In [ ]:
# SETUP
!pip install -q edge-tts
import json, subprocess, edge_tts
from pathlib import Path
from IPython.display import display, Markdown, Audio

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
print('✅ Setup complete')

In [ ]:
# ============================================================
# SELECT SESSION
# ============================================================
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d / 'session_info.json').exists()])

if not sessions:
    print('❌ No sessions found! Run 02_Script_Writer first.')
else:
    print('📂 Available sessions:')
    for i, s in enumerate(sessions): print(f'   {i}: {s}')
    print()

# ⬇️ CHOOSE YOUR SESSION ⬇️
SESSION_NAME = sessions[-1] if sessions else ''  # Default: latest session

SESSION_DIR = BASE_DIR / SESSION_NAME
AUDIO_DIR = SESSION_DIR / 'audio'; AUDIO_DIR.mkdir(exist_ok=True)
print(f'✅ Using session: {SESSION_NAME}')

In [ ]:
# LOAD ESSAY
essay_file = SESSION_DIR / 'essay_script.json'
prompts_file = SESSION_DIR / 'prompts.json'
narrations = []
if essay_file.exists():
    with open(essay_file) as f: essay = json.load(f)
    print(f'📖 {essay.get("title","Untitled")}')
    sn = 2
    for ch in essay.get('chapters',[]):
        for sec in ch.get('sections',[]):
            text = sec.get('narration','')
            if text:
                narrations.append({'scene':sn,'narration':text,'mood':sec.get('mood','')})
                print(f'   Scene {sn}: {len(text.split())}w')
            sn += 1
    print(f'   {len(narrations)} narrations')
else: print('❌ No essay found!')

In [ ]:
# VOICE
VOICES = {'female_us':'en-US-AriaNeural','male_us':'en-US-GuyNeural','female_uk':'en-GB-SoniaNeural','male_uk':'en-GB-RyanNeural','documentary':'en-US-GuyNeural','dramatic':'en-GB-RyanNeural','energetic':'en-US-JennyNeural'}
CHOSEN_VOICE = 'documentary'
voice_name = VOICES[CHOSEN_VOICE]
print(f'🎙️ {CHOSEN_VOICE} ({voice_name})')

In [ ]:
# GENERATE
async def gen_tts(text, path, voice, rate='+0%'):
    await edge_tts.Communicate(text, voice, rate=rate).save(str(path))

def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True,timeout=10).stdout.strip())
    except: return 5.0

print(f'🎙️ Generating {len(narrations)} narrations...')
audio_files = []
for n in narrations:
    out = AUDIO_DIR / f'narration_{n["scene"]:03d}.mp3'
    print(f'   Scene {n["scene"]}: {len(n["narration"].split())}w...', end=' ', flush=True)
    try:
        await gen_tts(n['narration'], out, voice_name)
        dur = get_dur(out)
        audio_files.append({'scene':n['scene'],'path':str(out),'duration':dur,'text':n['narration']})
        print(f'✅ {dur:.1f}s')
    except Exception as e: print(f'❌ {e}')

with open(SESSION_DIR / 'audio_manifest.json', 'w') as f: json.dump({'voice':voice_name,'files':audio_files}, f, indent=2)
total_dur = sum(a['duration'] for a in audio_files)
print(f'\n✅ {len(audio_files)} files, {total_dur:.1f}s total')

# Update prompts with actual durations
if prompts_file.exists():
    with open(prompts_file) as f: prompts = json.load(f)
    dur_map = {a['scene']:a['duration'] for a in audio_files}
    for p in prompts:
        if p['scene'] in dur_map: p['duration'] = max(3, min(10, int(dur_map[p['scene']])+1))
    with open(prompts_file,'w') as f: json.dump(prompts,f,indent=2)
    print('✅ Updated prompts.json with narration durations')

In [ ]:
# PREVIEW
for a in audio_files:
    display(Markdown(f'### Scene {a["scene"]} ({a["duration"]:.1f}s)\n> {a["text"]}'))
    if Path(a['path']).exists(): display(Audio(a['path'], autoplay=False))
    display(Markdown('---'))
print('💡 Try different voices by changing CHOSEN_VOICE!')

---
Next: **05_Generate** (videos) then **06_Assemble** (final).

*ROTBOTS Workshop — LI-MA 2026*